# Example of data import from Omnipose output, processing, and curation, with no cell linking

__Steps__:
1. Download zip folder for a given experiment/condition and extract the data. In this example, we will process data from 'CJW7020_Tac1_1.2uM.zip'
2. We implement 'import_processing_no_linking_code.py' to apply background subtraction, and extract single cell features from Omnipose masks. Curation based on phase and fluorescence intensities distribution in the cell is applied at this step, as well as SCF* as defined in the Methods.

__Requirements on how to run__:
1. This notebook assumes the Omnipose-processed extracted file data folder, here 'CJW7020_Tac1_1.2uM', lives in the same folder as the notebook

__Code underlying data and figures__:
- Single-cell data in PDMS microwells, strains CJW7020, CJW7859, with fluor AMPs, colocalization analysis with ribosome/HU
- Figures 4C

### 1. Run 'import_processing_no_linking_code.py'

Run time: < 1hr per dataset (~30-40 positions), generally depends on amount of detected cells form OmniSegger

In [5]:
import numpy as np
import pandas as pd
from skimage.measure import regionprops, find_contours
from scipy.stats import spearmanr
from scipy import ndimage
import matplotlib.pyplot as plt
import os
import imageio.v3 as iio


def extract_cell_data(mask, phase, fluor1, fluor2, cell_ch_dict, frame_number, frame_interval, cell_id_counter,
                      pos, exp_name, rep, px_size=0.065841, pad=5, show=False, offs_x=20, offs_y=20):
    data = []
    props = regionprops(mask)
    for prop in props:
        centroid = prop.centroid
        minr, minc, maxr, maxc = prop.bbox
        x1, x2, y1, y2 = minc - pad, maxc + pad, minr - pad, maxr + pad
        img_y, img_x = mask.shape
        if x1 < offs_x or y1 < offs_y or x2 > img_x - offs_x or y2 > img_y - offs_y:
            continue
        major_axis = prop.major_axis_length
        minor_axis = prop.minor_axis_length
        area = prop.area

        angle_deg = np.abs(np.degrees(prop.orientation))        

        if area < 500 or area > 2000:
            continue

        perimeter = prop.perimeter
        roundness = (4 * np.pi * area) / (perimeter**2)

        cell_mask = np.zeros(mask.shape)
        cell_mask[mask == prop.label] = prop.label              

        cropped_mask = cell_mask[y1:y2, x1:x2]
        cropped_phase = phase[y1:y2, x1:x2]
        cropped_fluor1 = fluor1[y1:y2, x1:x2]
        cropped_fluor2 = fluor2[y1:y2, x1:x2]

        if np.size(cropped_mask) != np.size(cropped_fluor1):
            continue
        mean_fluor1 = np.mean(cropped_fluor1[cropped_mask > 0])
        mean_phase = np.mean(cropped_phase[cropped_mask > 0])
        sd_phase = np.std(cropped_phase[cropped_mask > 0])
        mean_fluor2 = np.mean(cropped_fluor2[cropped_mask > 0])

        mask_ero1 = ndimage.binary_erosion(cropped_mask, iterations=1)
        mask_peri = np.logical_not(ndimage.binary_erosion(mask_ero1, iterations=2)) * mask_ero1
        phase_peri = cropped_phase[mask_peri > 0].ravel()
        mean_phase_peri = np.mean(phase_peri)
        mean_phase_peri_high = np.mean(phase_peri[phase_peri > np.percentile(phase_peri, 40)])

        if mean_phase_peri_high > 3300:
            continue

        cv_phase_peri = np.std(cropped_phase[mask_peri > 0].ravel()) / mean_phase_peri

        mask_eroded = mask_ero1
        fluor1_roi = cropped_fluor1[mask_eroded > 0].ravel()
        fluor2_roi = cropped_fluor2[mask_eroded > 0].ravel()

        percentile_threshold = 50
        percentile_threshold_2 = 90
        threshold1 = np.percentile(fluor1_roi, percentile_threshold)
        threshold2 = np.percentile(fluor2_roi, percentile_threshold)
        fluor1_norm = fluor1_roi / np.max(fluor1_roi)
        fluor2_norm = fluor2_roi / np.max(fluor2_roi)
        threshold3 = np.percentile(fluor1_norm + fluor2_norm, percentile_threshold_2)

        valid_idx = (fluor1_roi > threshold1) & (fluor2_roi > threshold2)
        fluor1_valid = fluor1_roi[valid_idx]
        fluor2_valid = fluor2_roi[valid_idx]

        valid_idx_2 = (fluor1_norm + fluor2_norm) > threshold3
        fluor1_valid_2 = fluor1_roi[valid_idx_2]
        fluor2_valid_2 = fluor2_roi[valid_idx_2]

        SCF_ori, _ = spearmanr(fluor1_roi, fluor2_roi)
        SCF, _ = spearmanr(fluor1_valid, fluor2_valid)
        SCF_2, _ = spearmanr(fluor1_valid_2, fluor2_valid_2)

        label_id = f"cell{prop.label:07d}"
        cell_id_counter.append(f"cell{len(cell_id_counter) + 1:07d}")
        if label_id in cell_id_counter:
            label_id = cell_id_counter[-1]
        cell_id = label_id + '_' + pos + '_' + exp_name

        data.append({
            "cell_id": cell_id, 'exp': exp_name, 'pos': pos, 'rep': rep,
            "frame": frame_number, "time_min": frame_number * frame_interval,
            "centroid_x": centroid[1], "centroid_y": centroid[0], 'roundness': roundness,
            "bbox_x_min": x1, "bbox_y_min": y1, "bbox_x_max": x2, "bbox_y_max": y2,
            "r_off": np.array([x1 + 1, y1 + 1]), 'rcm': np.array([centroid[1], centroid[0]]),
            'angle': angle_deg, "area": area, 'area_um': area * (px_size**2),
            "major_axis_length": major_axis, "minor_axis_length": minor_axis,
            'mean_phase': mean_phase, 'sd_phase': sd_phase,
            'mean_fluor1': mean_fluor1, 'mean_fluor2': mean_fluor2,
            'SCF_ori': SCF_ori, 'SCF': SCF,
        })
        cell_ch_dict[cell_id] = cropped_mask, cropped_phase, cropped_fluor1, cropped_fluor2

        if show:
            contours_ero = find_contours(mask_eroded, level=0.5)          
            contours = find_contours(cropped_mask, level=0.5)
            fig = plt.figure(figsize=(25, 10))
            fig.suptitle(cell_id + ', frame = ' + str(frame_number))   
            ax = fig.add_subplot(2, 4, 1); plt.title('mean phase = ' + str(round(mean_phase, 1))); plt.imshow(cropped_phase)
            for c in contours: plt.plot(c[:, 1], c[:, 0], linewidth=1, color='yellow', linestyle='--')
            fig.add_subplot(2, 4, 2); plt.title('Area = ' + str(area) + ', roundness = ' + str(round(roundness, 2))); plt.imshow(cropped_mask)
            fig.add_subplot(2, 4, 3); plt.title('mean_phase_peri_high = ' + str(round(mean_phase_peri_high, 2))); plt.imshow(mask_peri)
            ax = fig.add_subplot(2, 4, 4); plt.title('mean fluor1 = ' + str(round(mean_fluor1, 1))); plt.imshow(cropped_fluor1)
            for c in contours_ero: plt.plot(c[:, 1], c[:, 0], linewidth=1, color='yellow', linestyle='--')
            ax = fig.add_subplot(2, 4, 5); plt.title('mean fluor2 = ' + str(round(mean_fluor2, 1))); plt.imshow(cropped_fluor2 * cropped_mask)
            for c in contours_ero: plt.plot(c[:, 1], c[:, 0], linewidth=1, color='yellow', linestyle='--')
            fig.add_subplot(2, 4, 6); plt.title('SCF_ori = ' + str(round(SCF_ori, 1)))
            plt.scatter(fluor1_roi, fluor2_roi, alpha=0.5); plt.scatter(fluor1_valid, fluor2_valid, c='orange', alpha=0.5)
            plt.scatter(fluor1_valid_2, fluor2_valid_2, c='green', alpha=0.5); plt.xlabel('Fluor1'); plt.ylabel('Fluor2')
            fig.add_subplot(2, 4, 7); plt.title('SCF = ' + str(round(SCF, 1))); plt.scatter(fluor1_valid, fluor2_valid); plt.xlabel('Fluor1'); plt.ylabel('Fluor2')
            fig.add_subplot(2, 4, 8); plt.title('SCF_2 = ' + str(round(SCF_2, 1))); plt.scatter(fluor1_valid_2, fluor2_valid_2); plt.xlabel('Fluor1'); plt.ylabel('Fluor2')
            plt.tight_layout(); plt.show()

    df = pd.DataFrame(data)
    return df, cell_ch_dict, cell_id_counter

In [10]:
'''No-linking single-cell feature extraction — single experiment, positions only.
OmniSegger drift-correction + segmentation already done; extracts static per-cell
features (area, mean fluor, SCF) frame by frame, without cell linking.'''

"""
Import analysis functions library
"""
import sys
from pathlib import Path
sys.path.append(str(Path("../../2_singlecell_microwells").resolve()))
from analysis_functions_library import *

px_size = 0.065841
offs_x = 20
offs_y = 20
save = True
show = False
fr = 10    # [min] frame interval

ch_list = ['phase', 'fluor1', 'fluor2']         # this dataset has 2 fluor channels (no fluor3)

experiment_path = "CJW7020_Tac1_1.2uM"          # extracted archive folder, next to the notebook
exp_name = experiment_path
rep_name = "rep1"
pos_list = sorted([s for s in os.listdir(experiment_path) if s.startswith('xy')])


# read fr_inj from the notes 
notes_file = next(Path(experiment_path).glob("notes_*.txt"))
print(notes_file.read_text())   # check when AMP was injected 'frame xx/yy'

250122, CJW7020

37degC, M9GluCAAT (0.2% Glu, 0.4% CAA)

FR = 10 min interval 

CellTak:
- 50uL, 5min, of CellTak. 3x MQ wash.
- 0.5uL, OD=0.1, 5min, in M9GluCAAT. Quick medium wash.

Treatments/pos:
xy31-60: Tachy, 1.2 uM (+5% 5T-Tachy)--> injected at frame 3/4



In [11]:
fr_inj = 3

In [13]:

count = 0
for pos in pos_list:
    folder_path = experiment_path + '/' + pos
    mask_path = folder_path + '/masks'
    if not os.path.exists(mask_path):
        continue
    masks_list = os.listdir(mask_path)
    print(pos)

    paths_dict = {ch: folder_path + '/' + ch for ch in ch_list}
    file_list_dict = {ch: os.listdir(paths_dict[ch]) for ch in ch_list}
    phase_list, phase_path = file_list_dict['phase'], paths_dict['phase']
    fluor1_list, fluor1_path = file_list_dict['fluor1'], paths_dict['fluor1']
    fluor2_list, fluor2_path = file_list_dict['fluor2'], paths_dict['fluor2']

    cell_ch_dict = {}
    cell_features_all_df = pd.DataFrame()
    cell_id_counter = []

    print('..Subtract background and extract cell features at pos ' + pos + ' for experiment ' + exp_name)
    for k in range(len(masks_list)):
        if k % 20 == 0:
            print(k)
        mask_temp = iio.imread(mask_path + '/' + masks_list[k])
        phase_temp = iio.imread(phase_path + '/' + phase_list[k])
        fluor1_temp = iio.imread(fluor1_path + '/' + fluor1_list[k])
        fluor2_temp = iio.imread(fluor2_path + '/' + fluor2_list[k])

        bs = int(fluor1_temp.shape[0] / 16)
        fluor1_bkg_sub, _, _ = local_bkg_sub_cp(fluor1_temp, mask_temp, pos, exp_name, frame=str(k), box_size=bs, dilations=15, sigma_=60, show=False)
        fluor2_bkg_sub, _, _ = local_bkg_sub_cp(fluor2_temp, mask_temp, pos, exp_name, frame=str(k), box_size=bs, dilations=15, sigma_=60, show=False)

        df_temp, cell_ch_dict, cell_id_counter = extract_cell_data(
            mask_temp, phase_temp, fluor1_bkg_sub, fluor2_bkg_sub, cell_ch_dict, k, fr,
            cell_id_counter, pos, exp_name, rep_name, px_size=px_size,
            pad=5, show=show, offs_x=offs_x, offs_y=offs_y)
        df_temp['fr_inj'] = fr_inj
        cell_features_all_df = pd.concat([cell_features_all_df, df_temp])

    if save and len(cell_features_all_df) > 0:
        save_dict_path = experiment_path + '/output'
        if not os.path.exists(save_dict_path):
            os.makedirs(save_dict_path)
        cell_ch_dict_name = pos + '_' + exp_name + '_cell_ch_bkg_sub'
        df_basename = pos + '_' + exp_name + '_cell_features_df'
        with open(os.path.join(save_dict_path, cell_ch_dict_name + '.pkl'), 'wb') as pf:
            pickle.dump(cell_ch_dict, pf)
        cell_features_all_df.to_pickle(save_dict_path + '/' + df_basename + '.pkl')
        n = cell_features_all_df['cell_id'].unique().size
        count += n
        print('At position ' + pos + ' there are ' + str(n) + ' cells')

print('In total there are ' + str(count) + ' cells')

xy31
..Subtract background and extract cell features at pos xy31 for experiment CJW7020_Tac1_1.2uM
0
At position xy31 there are 176 cells
xy32
..Subtract background and extract cell features at pos xy32 for experiment CJW7020_Tac1_1.2uM
0
At position xy32 there are 155 cells
xy33
..Subtract background and extract cell features at pos xy33 for experiment CJW7020_Tac1_1.2uM
0
At position xy33 there are 169 cells
xy34
..Subtract background and extract cell features at pos xy34 for experiment CJW7020_Tac1_1.2uM
0
At position xy34 there are 196 cells
xy35
..Subtract background and extract cell features at pos xy35 for experiment CJW7020_Tac1_1.2uM
0
At position xy35 there are 231 cells
xy36
..Subtract background and extract cell features at pos xy36 for experiment CJW7020_Tac1_1.2uM
0
At position xy36 there are 224 cells
xy37
..Subtract background and extract cell features at pos xy37 for experiment CJW7020_Tac1_1.2uM
0
At position xy37 there are 352 cells
xy38
..Subtract background and ext

__Output:__ 
- Per-position cell features dataframes, ending with '_cell_features_df'
- Per-position dictionaries containing cropped single cell images of all background subtracted channels + Omnipose masks

All saved in /output folder in the same directory